# Ingestion — BGE-M3 on CUDA

The aim of this notebook is to transform legal documents into vector embeddings and upload them to Pinecone.

The embedding model used is **BGE-M3**.

## Setup

1. Attach a GPU-backed kernel.
2. Upload the `Contest_Data/` directory.
3. Upload `Apikey.env`, containing `PINECONE_API_KEY`, to the notebook’s working directory.

## Pinecone index

The `legal-rag` index must have **1,024 dimensions** to match BGE-M3’s embedding output.

## Flow

1. Extract the optional ZIP safely.
2. Validate local configuration and the Pinecone index.
3. Parse and validate every JSON without writing remotely.
4. Split content into overlapping token chunks and enrich each chunk.
5. Preview the resulting records.
6. Optionally clear the target namespace, embed batches, and upsert them.

Chunk size: **1,500 tokens**; overlap: **150 tokens**.

### Example

A Slovenian inheritance JSON becomes metadata such as `country=Slovenia`, `doc_type=Civil Codes`, and `law=Inheritance`. If its content is longer than 1,500 tokens, it becomes multiple overlapping chunks. Each vector embeds a compact header, every available metadata field, and the chunk text; Pinecone stores the same metadata, the raw chunk, its stable ID, and its citation label for later filtered retrieval.


## 1.Extract Zip 

In [1]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("Contest_Data.zip")
DATA_PATH = Path("Contest_Data")

if ZIP_PATH.exists() and not DATA_PATH.exists():
    destination = Path.cwd().resolve()
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)
    print(f"Extracted {ZIP_PATH} into {destination}")
else:
    print("Dataset already present; ZIP extraction skipped.")

Dataset already present; ZIP extraction skipped.


## 2.Imports

In [2]:
import hashlib
import json
import os
import time
from collections import Counter
from pathlib import Path
import re

import torch
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from tqdm.auto import tqdm  

## 3.Config

In [3]:
load_dotenv("Apikey.env")

DOCS_DIR = Path("Contest_Data")
INDEX_NAME = "legal-rag"
INDEX_DIMENSION = 1024
INDEX_METRIC = "cosine"
NAMESPACE = ""  # Pinecone default namespace

COUNTRY_MAP = {"italy": "Italy", "estonia": "Estonia", "slovenia": "Slovenia"}


PINECONE_METADATA_LIMIT_BYTES = 40 * 1024
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 150

RESET_NAMESPACE_BEFORE_UPSERT = True

EMBED_MODEL_NAME = "BAAI/bge-m3"

if CHUNK_SIZE <= 0 or not 0 <= CHUNK_OVERLAP < CHUNK_SIZE:
    raise ValueError("Require CHUNK_SIZE > 0 and 0 <= CHUNK_OVERLAP < CHUNK_SIZE")
if not DOCS_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DOCS_DIR.resolve()}")

## 4.Startup validation, device check, model + client init

In [5]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise SystemExit("Missing PINECONE_API_KEY — add it to Apikey.env")

pc = Pinecone(api_key=PINECONE_API_KEY)
index_description = pc.describe_index(INDEX_NAME)
actual_dimension = index_description.dimension
actual_metric = str(index_description.metric).lower()
if actual_dimension != INDEX_DIMENSION:
    raise ValueError(
        f"Index {INDEX_NAME!r} has dimension {actual_dimension}; "
        f"expected {INDEX_DIMENSION}."
    )
if actual_metric != INDEX_METRIC:
    raise ValueError(
        f"Index {INDEX_NAME!r} uses metric {actual_metric!r}; "
        f"expected {INDEX_METRIC!r}."
    )
pine_index = pc.Index(INDEX_NAME)
print(f"Pinecone index validated: {INDEX_NAME} ({actual_dimension}D, {actual_metric})")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")
if device == "cpu":
    print("  [WARN] No CUDA GPU detected — check this kernel is attached "
          "to a GPU machine. BGE-M3 will be slow on CPU (568M params).")
else:
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

EMBED_MODEL = SentenceTransformer(EMBED_MODEL_NAME, device=device)
model_dimension = EMBED_MODEL.get_sentence_embedding_dimension()
if model_dimension != INDEX_DIMENSION:
    raise ValueError(
        f"Model outputs {model_dimension} dimensions; expected {INDEX_DIMENSION}."
    )

Pinecone index validated: legal-rag (1024D, cosine)
Embedding device: cuda
  GPU: NVIDIA RTX A4000


## 5.Detection of Doc Type

It's purely structural ("Legal Cases" vs "Civil Codes"), law ("Divorce"/"Inheritance") is derived from the folder name for civil codes, and read from each file's own metadata (passthrough, no alias resolution) for cases.

In [6]:
def detect_doc_type(folder_name: str):
    name = folder_name.lower()
    if "case" in name:
        return "Legal Cases"
    if "divorce" in name or "inheritance" in name:
        return "Civil Codes"
    return None


def normalize_law(raw_law: str, doc_type: str, folder_name: str):
    if doc_type == "Civil Codes":
        name = folder_name.lower()
        if "divorce" in name:
            return "Divorce"
        if "inheritance" in name:
            return "Inheritance"
        return "Unknown"
    return (raw_law or "").strip() or "Unknown"

## 6.MetaData Prepa
Prepares document metadata for Pinecone and embedding.
First, metadata values are converted into Pinecone-compatible types.
Then every available field is combined into a readable `key: value` line.
The only excluded field is `text`, because the raw chunk is appended separately.

In [7]:
def flatten_metadata(raw):
    # None -> "" (Pinecone rejects null metadata values); bool/int/float
    # kept native (so numeric filters like $gt/$lt stay possible later);
    # list -> list of strings (Pinecone arrays must be homogeneous
    # strings); anything else -> str(value).
    flat = {}
    for k, val in raw.items():
        if val is None:
            flat[k] = ""
        elif isinstance(val, bool):
            flat[k] = val
        elif isinstance(val, (int, float)):
            flat[k] = val
        elif isinstance(val, list):
            flat[k] = [str(v) for v in val]
        else:
            flat[k] = str(val)
    return flat


# Include every available metadata field in the embedding text.
# `text` is the only exception because the raw chunk is appended separately.
_METADATA_LINE_SKIP_KEYS = {"text"}
_METADATA_LINE_PLACEHOLDER_VALUES = {"", "no data", "not specified", "n/a", "unknown"}


def _normalize_article_label_item(value) -> str:
    item = str(value or "").strip()
    if not item:
        return ""

    # Canonicalize article prefixes at ingestion time, including the
    # duplicated form occasionally found in source metadata. Examples:
    # Article 210 / ARTICLE 210 / Art 210 / Art. Article 210 -> Art. 210
    match = re.fullmatch(
        r"(?i)(?:art(?:icle)?\.?\s*)+(\d+[a-zA-Z\-]*(?:\s*\(\s*\d+[a-zA-Z]*\s*\))?)",
        item,
    )
    if not match:
        return item

    article_number = re.sub(r"\s+", "", match.group(1))
    return f"Art. {article_number}"


def normalize_article_metadata(value) -> str:
    values = value if isinstance(value, list) else [value]
    items = [_normalize_article_label_item(item) for item in values]
    return ", ".join(item for item in items if item)


def build_metadata_line(meta: dict) -> str:
    parts = []
    for key, value in meta.items():
        if key in _METADATA_LINE_SKIP_KEYS or value in (None, [], {}):
            continue
        display = ", ".join(value) if isinstance(value, list) else value
        parts.append(f"{key}: {display}")
    return "; ".join(parts)


def label_from_heading(content: str) -> str:
    first_line = next((line.strip() for line in content.splitlines() if line.strip()), "")
    match = re.search(
        r"(?i)\b(?:article|art\.?|section|§{1,2}|act)\s*[A-Z]*\s*\d+[\w()./-]*",
        first_line,
    )
    return match.group(0).strip() if match else ""

## 7. Chunking and embedding-text helpers — definitions only

This cell only defines the functions used for token chunking, stable IDs, citation labels, and enriched embedding text. It does not process documents yet. Actual chunk creation happens later inside `run_ingestion`.

In [8]:
def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
):
    token_ids = EMBED_MODEL.tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    if len(token_ids) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(token_ids):
        end = min(start + chunk_size, len(token_ids))

        chunk = EMBED_MODEL.tokenizer.decode(
            token_ids[start:end],
            skip_special_tokens=True,
        ).strip()

        chunks.append(chunk)

        if end == len(token_ids):
            break

        start += step

    return chunks


def _as_header_str(value) -> str:
    if isinstance(value, list):
        return ", ".join(value)
    return str(value) if value else ""


def build_embedding_text(chunk: str, meta: dict) -> str:
    # Prepend a compact header + the extra-metadata line to the chunk
    # before embedding, e.g.:
    #   [Estonia | Legal Cases | Divorce | citation label]
    #   <all available metadata as key: value pairs>
    #   <chunk text>
    country  = _as_header_str(meta.get("country", ""))
    doc_type = _as_header_str(meta.get("doc_type", ""))
    law      = _as_header_str(meta.get("law", ""))
    citation = _as_header_str(meta.get("citation_label", ""))
    meta_line = build_metadata_line(meta)

    header_parts = [p for p in [country, doc_type, law, citation] if p]
    header = "[" + " | ".join(header_parts) + "]" if header_parts else ""

    parts = [p for p in [header, meta_line, chunk] if p]
    return "\n".join(parts)


def make_chunk_id(source: str, chunk_index: int) -> str:
    # Creates a stable ID from the source path and chunk position.
    # Pinecone requires a unique ID for every vector, and this distinguishes
    # multiple chunks from the same document.
    raw = f"{source}::{chunk_index}"
    return hashlib.md5(raw.encode()).hexdigest()


def build_citation_label(meta: dict, content: str) -> str:
    case_id = str(meta.get("CASE_ID") or "").strip()
    if case_id:
        return case_id

    article_label = normalize_article_metadata(
        meta.get("civil_codes_used")
    )
    if article_label:
        return article_label

    heading_label = normalize_article_metadata(label_from_heading(content))
    if heading_label:
        return heading_label

    source_name = Path(str(meta.get("source") or "document")).stem
    return source_name

## 8. Document loading and validation — definition only

This cell defines how JSON files are read, validated, classified, and normalized. It does not perform chunking, embedding, or Pinecone writes.

In [9]:
def load_documents(docs_dir):
    docs, errors, unknown_law = [], [], []
    tally = Counter()

    all_files = sorted(docs_dir.rglob("*.json"))

    for f in tqdm(all_files, desc="Loading documents", unit="doc"):
        relative_parts = f.relative_to(docs_dir).parts
        if len(relative_parts) != 3:
            errors.append(f"{f}: expected country/category/file.json layout")
            continue
        folder_country = relative_parts[0].lower()
        folder_type = relative_parts[1]
        country  = COUNTRY_MAP.get(folder_country)
        doc_type = detect_doc_type(folder_type)
        if not country or not doc_type:
            errors.append(f"{f}: unrecognised folder")
            continue

        try:
            doc = json.loads(f.read_text(encoding="utf-8"))
        except Exception as e:
            errors.append(f"{f.name}: {e}")
            continue

        content = (doc.get("content") or "").strip()
        if not content:
            errors.append(f"{f.name}: empty content")
            continue

        meta = flatten_metadata(doc.get("metadata", {}))
        meta.update({"country": country, "doc_type": doc_type, "source": str(f)})

        law = normalize_law(meta.get("law", ""), doc_type, folder_type)
        meta["law"] = law
        if law == "Unknown":
            unknown_law.append(f"{f.name} (raw law = {doc.get('metadata', {}).get('law')!r})")

        docs.append({"content": content, "metadata": meta})
        tally[(country, doc_type, law)] += 1

    print(f"Loaded: {len(docs)} | Errors: {len(errors)}")
    for e in errors:
        print(f"  [WARN] {e}")

    if unknown_law:
        print(f"\n  [WARNING] {len(unknown_law)} file(s) with unrecognised 'law' "
              f"(ingested as law='Unknown'; Divorce/Inheritance agents will NOT "
              f"find them until fixed):")
        for u in unknown_law:
            print(f"    - {u}")

    print("\n  Summary by (country, type, law):")
    for (c, t, l), n in sorted(tally.items()):
        print(f"    {c:8} | {t:22} | {l:12} : {n} docs")

    if errors:
        raise ValueError(f"Dataset validation failed for {len(errors)} file(s).")
    if unknown_law:
        raise ValueError(f"Unknown law in {len(unknown_law)} file(s).")
    if not docs:
        raise ValueError("No valid JSON documents found.")
    return docs

## 9. RUN: load and validate documents

**Checkpoint.** Read the report before running the next section — check `Errors`, the `Unknown` law list, and whether the per-country/type/law counts look right, before spending GPU time and writing anything to Pinecone.

In [10]:
docs = load_documents(DOCS_DIR)

Loading documents:   0%|          | 0/1473 [00:00<?, ?doc/s]

Loaded: 1473 | Errors: 0

  Summary by (country, type, law):
    Estonia  | Civil Codes            | Divorce      : 50 docs
    Estonia  | Civil Codes            | Inheritance  : 49 docs
    Estonia  | Legal Cases            | Divorce      : 91 docs
    Estonia  | Legal Cases            | Inheritance  : 31 docs
    Italy    | Civil Codes            | Divorce      : 38 docs
    Italy    | Civil Codes            | Inheritance  : 303 docs
    Italy    | Legal Cases            | Divorce      : 129 docs
    Italy    | Legal Cases            | Inheritance  : 341 docs
    Slovenia | Civil Codes            | Divorce      : 31 docs
    Slovenia | Civil Codes            | Inheritance  : 209 docs
    Slovenia | Legal Cases            | Divorce      : 101 docs
    Slovenia | Legal Cases            | Inheritance  : 100 docs


## 10. Record construction and Pinecone helpers — definitions only

These functions create chunks and vector records, preview them, enforce the metadata limit, optionally reset the namespace, and perform batched upserts. Nothing is written until the next cell calls `run_ingestion`.

In [11]:
def build_chunk_records(docs):
    # One entry per source document, each holding its list of chunk
    # records.
    per_doc = []
    for d in docs:
        content = d["content"]
        chunks = chunk_text(content)
        source = d["metadata"].get("source", "")
        chunk_records = []
        for ci, chunk in enumerate(chunks):
            meta = dict(d["metadata"])
            meta["text"] = chunk
            meta["chunk_index"] = ci
            meta["n_chunks"] = len(chunks)
            meta["citation_label"] = build_citation_label(meta, content)
            chunk_records.append({
                "id": make_chunk_id(source, ci),
                "embed_text": build_embedding_text(chunk, meta),
                "metadata": meta,
            })
        per_doc.append({"source": source, "chunks": chunk_records})
    return per_doc


def metadata_size(meta: dict) -> int:
    return len(json.dumps(meta, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def fit_metadata(meta: dict) -> dict:
    # Pinecone caps metadata at 40KB/vector. If exceeded, truncate only
    # the "text" field (not the other metadata) to fit.
    size = metadata_size(meta)
    if size <= PINECONE_METADATA_LIMIT_BYTES:
        return meta
    meta = dict(meta)
    text = meta.get("text", "")
    low, high = 0, len(text)
    while low < high:
        middle = (low + high + 1) // 2
        meta["text"] = text[:middle]
        if metadata_size(meta) <= PINECONE_METADATA_LIMIT_BYTES:
            low = middle
        else:
            high = middle - 1
    meta["text"] = text[:low]
    if metadata_size(meta) > PINECONE_METADATA_LIMIT_BYTES:
        raise ValueError("Non-text metadata alone exceeds Pinecone's metadata limit.")
    return meta


def flatten_chunk_records(per_doc):
    return [chunk for document in per_doc for chunk in document["chunks"]]


def preview_records(records, count=2):
    print(f"Prepared {len(records)} chunks. Preview:")
    for record in records[:count]:
        meta = record["metadata"]
        print({
            "id": record["id"],
            "citation_label": meta["citation_label"],
            "country": meta["country"],
            "doc_type": meta["doc_type"],
            "law": meta["law"],
            "chunk": f"{meta['chunk_index'] + 1}/{meta['n_chunks']}",
            "text_preview": meta["text"][:160],
        })


def reset_namespace():
    target = NAMESPACE or "<default>"
    print(f"Clearing Pinecone namespace {target}...")
    pine_index.delete(delete_all=True, namespace=NAMESPACE)
    deadline = time.monotonic() + 60
    while time.monotonic() < deadline:
        stats = pine_index.describe_index_stats()
        namespaces = getattr(stats, "namespaces", {}) or {}
        namespace_stats = namespaces.get(NAMESPACE)
        if isinstance(namespace_stats, dict):
            remaining = namespace_stats.get("vector_count", 0)
        else:
            remaining = getattr(namespace_stats, "vector_count", 0) if namespace_stats else 0
        if remaining == 0:
            print("Namespace cleared.")
            return
        time.sleep(1)
    raise TimeoutError(f"Namespace {target} was not empty after 60 seconds.")


def upsert_records(records, batch_size=100):
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    n_batches = (len(records) + batch_size - 1) // batch_size
    for start in tqdm(
        range(0, len(records), batch_size),
        desc="Embedding + upserting to Pinecone",
        unit="batch",
        total=n_batches,
    ):
        batch = records[start:start + batch_size]
        # normalize_embeddings=True: BGE-M3's own usage docs recommend
        # normalised dense vectors for cosine-similarity retrieval. If
        # your Pinecone index metric isn't cosine, drop this.
        embeds = EMBED_MODEL.encode(
            [r["embed_text"] for r in batch],
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        vectors = []
        for i, r in enumerate(batch):
            meta = fit_metadata(dict(r["metadata"]))
            vectors.append({
                "id": r["id"],
                "values": embeds[i].tolist(),
                "metadata": meta,
            })
        pine_index.upsert(vectors=vectors, namespace=NAMESPACE)
    print(f"Done: upserted {len(records)} vectors.")


def run_ingestion(docs, batch_size=100, reset=False):
    per_doc = build_chunk_records(docs)
    records = flatten_chunk_records(per_doc)
    preview_records(records)
    if reset:
        reset_namespace()
    upsert_records(records, batch_size=batch_size)
    return records

## 11. RUN: chunk, preview, reset, embed and upsert

Only run this after checking the validation report. The cell previews the first records and then writes to Pinecone. When `RESET_NAMESPACE_BEFORE_UPSERT=True`, it clears only the configured namespace and waits until it is empty before uploading the new vectors.

In [12]:
records = run_ingestion(
    docs,
    batch_size=100,
    reset=RESET_NAMESPACE_BEFORE_UPSERT,
)

Prepared 1473 chunks. Preview:
{'id': 'e043673f109ec344f980978f1cae6d45', 'citation_label': 'Art. 210', 'country': 'Estonia', 'doc_type': 'Civil Codes', 'law': 'Divorce', 'chunk': '1/1', 'text_preview': '210.  \nImplementation of Act\n(1) This Act extends to all the family law relationships which have arisen by the time this Act enters into force unless otherwise '}
{'id': 'a7cd30621aeaf9ca4145a7847f38b916', 'citation_label': 'Art. 211', 'country': 'Estonia', 'doc_type': 'Civil Codes', 'law': 'Divorce', 'chunk': '1/1', 'text_preview': '211.  \nMarital property right\n (1) In the case of marriages contracted before entry into force of this Act, the provisions of this Act regarding jointness of pr'}
Clearing Pinecone namespace <default>...
Namespace cleared.


Embedding + upserting to Pinecone:   0%|          | 0/15 [00:00<?, ?batch/s]

Done: upserted 1473 vectors.
